# MIRIPVIR25 - Draft 00 - Data processing

## Unique identifiers

We will use this extremely simple approach to create unique identifiers: creating a self-incrementing index based on different counters.

In [1]:
import pandas as pd
import requests
import json
import seaborn as sns 

In [2]:
class Taxon:
    def __init__(self):
        self.id = 0
    def __call__(self):
        out = "{:012}".format(self.id)
        self.id +=1
        return out

In [3]:
taxonids = Taxon()
hitcodes = Taxon()
hostids = Taxon()

## Step 1 - Libraries and sampling sites



In [4]:
libraries = pd.read_csv("data/mcleish24-TableS2.csv", sep=';')
libraries

,Library_code,Host_taxon,Habitat,Collection_code,No_extracts
0,PV001,Amaranthus sp,Crop,M1V,8
1,PV002,Convolvulus arvensis,Crop,M1V,11
2,PV003,Cucumis melo,Crop,M1V,13
3,PV003bgi,Cucumis melo,Crop,M1V,13
4,PV004bgi,Cyperus longus,Crop,M1V,7
...,...,...,...,...,...
318,PV586,Fumaria parviflora,Crop,H1P,8
319,PV587,Hirschfeldia incana,Crop,H1P,8
320,PV588,Hordeum vulgare,Crop,H1P,8
321,PV589,Hordeum vulgare,Crop,H1P,8


In [5]:
sampling = pd.read_csv("data/mcleish24-TableS1.csv", sep=';')
sampling = sampling.groupby(['Site_code', 'Collection_code', 'Location', 'Longitude', 'Latitude'], as_index = False)['Date'].apply(list)
sampling

,Site_code,Collection_code,Location,Longitude,Latitude,Date
0,C1,C1F,Aranjuez,-3.593308,40.051302,[1/2/16]
1,C1,C3F,Aranjuez,-3.593308,40.051302,[30/1/17]
2,C2,C2F,Aranjuez,-3.599064,40.043193,[1/2/16]
3,C2,C4F,Aranjuez,-3.599064,40.043193,[30/1/17]
4,E1,E1F,Aranjuez,-3.500323,40.059138,"[19/11/15, 10/11/16]"
5,E1,E1P,Aranjuez,-3.500323,40.059138,"[23/5/16, 3/5/17]"
6,E2,E2F,San Martín de la Vega,-3.536191,40.234966,"[19/11/15, 10/11/16]"
7,E2,E2P,San Martín de la Vega,-3.536191,40.234966,"[23/5/16, 10/5/17]"
8,E3,E3F,Ambite,-3.196057,40.089637,"[1/2/16, 10/11/16]"
9,E3,E3P,Ambite,-3.196057,40.089637,"[31/5/16, 3/5/17]"


In [6]:
libraries['Collection_code'] = libraries['Collection_code'].apply(lambda x: x.split("_"))
libraries['Collection_code']

0      [M1V]
1      [M1V]
2      [M1V]
3      [M1V]
4      [M1V]
       ...  
318    [H1P]
319    [H1P]
320    [H1P]
321    [H1P]
322    [Z1V]
Name: Collection_code, Length: 323, dtype: object

In [7]:
tmp = libraries.drop_duplicates(subset=['Library_code'])[['Library_code', 'Collection_code']].to_dict(orient='records')
library_collection_code = []
for item in tmp:
    for collection_code in item['Collection_code']:
        library_collection_code.append({"Library_code": item['Library_code'], "Collection_code": collection_code})
library_collection_code = pd.DataFrame.from_records(library_collection_code)
library_collection_code

,Library_code,Collection_code
0,PV001,M1V
1,PV002,M1V
2,PV003,M1V
3,PV003bgi,M1V
4,PV004bgi,M1V
...,...,...
330,PV586,H1P
331,PV587,H1P
332,PV588,H1P
333,PV589,H1P


In [8]:
library_site = pd.merge(sampling[['Collection_code', 'Site_code']], library_collection_code, on='Collection_code')
library_site = library_site[['Site_code', 'Library_code']].to_dict(orient='records')


In [9]:
sampling

,Site_code,Collection_code,Location,Longitude,Latitude,Date
0,C1,C1F,Aranjuez,-3.593308,40.051302,[1/2/16]
1,C1,C3F,Aranjuez,-3.593308,40.051302,[30/1/17]
2,C2,C2F,Aranjuez,-3.599064,40.043193,[1/2/16]
3,C2,C4F,Aranjuez,-3.599064,40.043193,[30/1/17]
4,E1,E1F,Aranjuez,-3.500323,40.059138,"[19/11/15, 10/11/16]"
5,E1,E1P,Aranjuez,-3.500323,40.059138,"[23/5/16, 3/5/17]"
6,E2,E2F,San Martín de la Vega,-3.536191,40.234966,"[19/11/15, 10/11/16]"
7,E2,E2P,San Martín de la Vega,-3.536191,40.234966,"[23/5/16, 10/5/17]"
8,E3,E3F,Ambite,-3.196057,40.089637,"[1/2/16, 10/11/16]"
9,E3,E3P,Ambite,-3.196057,40.089637,"[31/5/16, 3/5/17]"


In [10]:
tmp = pd.merge(sampling, library_collection_code, on='Collection_code')
tmp = tmp.drop_duplicates(subset=['Library_code'])[['Library_code', 'Date']].to_dict(orient='records')
dates = []
for item in tmp:
    for date in item['Date']:
        dates.append({"Library_code": item['Library_code'], "Date": date})
dates

[{'Library_code': 'PV534', 'Date': '1/2/16'},
 {'Library_code': 'PV535', 'Date': '1/2/16'},
 {'Library_code': 'PV538', 'Date': '1/2/16'},
 {'Library_code': 'PV540', 'Date': '1/2/16'},
 {'Library_code': 'PV544', 'Date': '1/2/16'},
 {'Library_code': 'PV546', 'Date': '1/2/16'},
 {'Library_code': 'PV551', 'Date': '1/2/16'},
 {'Library_code': 'PV553', 'Date': '1/2/16'},
 {'Library_code': 'PV554', 'Date': '1/2/16'},
 {'Library_code': 'PV556', 'Date': '1/2/16'},
 {'Library_code': 'PV557', 'Date': '1/2/16'},
 {'Library_code': 'PV064', 'Date': '19/11/15'},
 {'Library_code': 'PV064', 'Date': '10/11/16'},
 {'Library_code': 'PV065', 'Date': '19/11/15'},
 {'Library_code': 'PV065', 'Date': '10/11/16'},
 {'Library_code': 'PV066', 'Date': '19/11/15'},
 {'Library_code': 'PV066', 'Date': '10/11/16'},
 {'Library_code': 'PV068', 'Date': '19/11/15'},
 {'Library_code': 'PV068', 'Date': '10/11/16'},
 {'Library_code': 'PV069', 'Date': '19/11/15'},
 {'Library_code': 'PV069', 'Date': '10/11/16'},
 {'Library_cod

In [11]:
sites_to_type = {
    "Crop": "CO_020:0000051",
    "Edge": "CO_020:0000054",
    "Wasteland": "CO_020:0000046",
    "Oak": "CO_020:0000045"
}

In [12]:
sampling

,Site_code,Collection_code,Location,Longitude,Latitude,Date
0,C1,C1F,Aranjuez,-3.593308,40.051302,[1/2/16]
1,C1,C3F,Aranjuez,-3.593308,40.051302,[30/1/17]
2,C2,C2F,Aranjuez,-3.599064,40.043193,[1/2/16]
3,C2,C4F,Aranjuez,-3.599064,40.043193,[30/1/17]
4,E1,E1F,Aranjuez,-3.500323,40.059138,"[19/11/15, 10/11/16]"
5,E1,E1P,Aranjuez,-3.500323,40.059138,"[23/5/16, 3/5/17]"
6,E2,E2F,San Martín de la Vega,-3.536191,40.234966,"[19/11/15, 10/11/16]"
7,E2,E2P,San Martín de la Vega,-3.536191,40.234966,"[23/5/16, 10/5/17]"
8,E3,E3F,Ambite,-3.196057,40.089637,"[1/2/16, 10/11/16]"
9,E3,E3P,Ambite,-3.196057,40.089637,"[31/5/16, 3/5/17]"


In [13]:
sites = pd.merge(
    pd.merge(sampling, library_collection_code, on='Collection_code'),
    libraries[['Library_code', 'Habitat']], on='Library_code'
)
sites['Habitat_type'] = sites['Habitat'].map(sites_to_type)
sites = sites.drop_duplicates(subset=['Site_code'])[['Site_code', 'Longitude', 'Latitude', 'Location', 'Habitat', 'Habitat_type']].to_dict(orient='records')
sites

[{'Site_code': 'C1',
  'Longitude': -3.593308,
  'Latitude': 40.051302,
  'Location': 'Aranjuez',
  'Habitat': 'Crop',
  'Habitat_type': 'CO_020:0000051'},
 {'Site_code': 'C2',
  'Longitude': -3.599064,
  'Latitude': 40.043193,
  'Location': 'Aranjuez',
  'Habitat': 'Crop',
  'Habitat_type': 'CO_020:0000051'},
 {'Site_code': 'E1',
  'Longitude': -3.500323,
  'Latitude': 40.059138,
  'Location': 'Aranjuez',
  'Habitat': 'Wasteland',
  'Habitat_type': 'CO_020:0000046'},
 {'Site_code': 'E2',
  'Longitude': -3.536191,
  'Latitude': 40.234966,
  'Location': 'San Martín de la Vega',
  'Habitat': 'Wasteland',
  'Habitat_type': 'CO_020:0000046'},
 {'Site_code': 'E3',
  'Longitude': -3.196057,
  'Latitude': 40.089637,
  'Location': 'Ambite',
  'Habitat': 'Wasteland',
  'Habitat_type': 'CO_020:0000046'},
 {'Site_code': 'E4',
  'Longitude': -3.131139,
  'Latitude': 40.494167,
  'Location': 'Mondéjar',
  'Habitat': 'Wasteland',
  'Habitat_type': 'CO_020:0000046'},
 {'Site_code': 'H1',
  'Longitude

In [14]:
libraries = libraries.drop_duplicates(subset=['Library_code'])[['Library_code', 'No_extracts']].to_dict(orient='records')
libraries

[{'Library_code': 'PV001', 'No_extracts': 8},
 {'Library_code': 'PV002', 'No_extracts': 11},
 {'Library_code': 'PV003', 'No_extracts': 13},
 {'Library_code': 'PV003bgi', 'No_extracts': 13},
 {'Library_code': 'PV004bgi', 'No_extracts': 7},
 {'Library_code': 'PV005bgi', 'No_extracts': 12},
 {'Library_code': 'PV006bgi', 'No_extracts': 19},
 {'Library_code': 'PV007bgi', 'No_extracts': 17},
 {'Library_code': 'PV008bgi', 'No_extracts': 11},
 {'Library_code': 'PV009', 'No_extracts': 13},
 {'Library_code': 'PV009bgi', 'No_extracts': 13},
 {'Library_code': 'PV010bgi', 'No_extracts': 10},
 {'Library_code': 'PV011', 'No_extracts': 7},
 {'Library_code': 'PV012bgi', 'No_extracts': 7},
 {'Library_code': 'PV013bgi', 'No_extracts': 9},
 {'Library_code': 'PV014', 'No_extracts': 8},
 {'Library_code': 'PV015bgi', 'No_extracts': 11},
 {'Library_code': 'PV016', 'No_extracts': 16},
 {'Library_code': 'PV017bgi', 'No_extracts': 12},
 {'Library_code': 'PV018bgi', 'No_extracts': 16},
 {'Library_code': 'PV020bgi

In [15]:
site_library = dict(
    library=libraries,
    library_dates=dates,
    site=sites, 
    library_site=library_site
)
with open("scratch/semantic-model/library.json", "w") as f:
    json.dump(site_library, f, indent=4, sort_keys=True)

## Step 2 - Hosts

In [16]:
libraries = pd.read_csv("data/mcleish24-TableS2.csv", sep=';')
libraries

,Library_code,Host_taxon,Habitat,Collection_code,No_extracts
0,PV001,Amaranthus sp,Crop,M1V,8
1,PV002,Convolvulus arvensis,Crop,M1V,11
2,PV003,Cucumis melo,Crop,M1V,13
3,PV003bgi,Cucumis melo,Crop,M1V,13
4,PV004bgi,Cyperus longus,Crop,M1V,7
...,...,...,...,...,...
318,PV586,Fumaria parviflora,Crop,H1P,8
319,PV587,Hirschfeldia incana,Crop,H1P,8
320,PV588,Hordeum vulgare,Crop,H1P,8
321,PV589,Hordeum vulgare,Crop,H1P,8


In [17]:
libraries[['Host_taxon', 'Library_code']]

,Host_taxon,Library_code
0,Amaranthus sp,PV001
1,Convolvulus arvensis,PV002
2,Cucumis melo,PV003
3,Cucumis melo,PV003bgi
4,Cyperus longus,PV004bgi
...,...,...
318,Fumaria parviflora,PV586
319,Hirschfeldia incana,PV587
320,Hordeum vulgare,PV588
321,Hordeum vulgare,PV589


In [18]:
def search_taxonomy(host_name):
    url = "https://api.ncbi.nlm.nih.gov/datasets/v2"
    url = f'{url}/taxonomy/taxon/{host_name}'.replace(' ', '%20')

    r = requests.get(url)
    try: 
        r.raise_for_status()
    except requests.HTTPError:
        return pd.NA
    try:
        u = str(r.json()['taxonomy_nodes'][0]['taxonomy']['tax_id'])
    except KeyError:
        return pd.NA
    except IndexError:
        return pd.NA
    return u

In [19]:
hosts = libraries.drop_duplicates(subset=['Host_taxon'])[['Host_taxon']]
hosts['Host_curated'] = hosts['Host_taxon'].apply(lambda x: x.replace(' sp', ' sp.'))
hosts['Host_curated'] = hosts['Host_curated'].apply(lambda x: " ".join(x.split(" ")[:2]))
species_taxid_map = []
for item in hosts['Host_curated'].unique().tolist():
    species_taxid_map.append(
        dict(species_name=item, taxid=search_taxonomy(item))
    )
hosts = pd.merge(hosts, pd.DataFrame.from_records(species_taxid_map), left_on='Host_curated', right_on='species_name')
hosts = hosts[['Host_taxon', 'taxid']]
hosts['Taxon_code'] = [hostids() for _ in range(len(hosts))]
hosts

,Host_taxon,taxid,Taxon_code
0,Amaranthus sp,3076757,000000000000
1,Convolvulus arvensis,4123,000000000001
2,Cucumis melo,3656,000000000002
3,Cyperus longus,76433,000000000003
4,Artemisia herba alba,<NA>,000000000004
...,...,...,...
114,Lithodora fruticosa,475890,000000000114
115,Origanum vulgare,39352,000000000115
116,Rhamnus lycioides,72169,000000000116
117,Astragalus sesameus,2014722,000000000117


In [20]:
hosts_library = pd.merge(hosts, libraries, on=['Host_taxon']).drop_duplicates(['Taxon_code', 'Library_code'], keep='first')[['Taxon_code', 'Library_code']].to_dict(orient='records')
hosts_library

[{'Taxon_code': '000000000000', 'Library_code': 'PV001'},
 {'Taxon_code': '000000000000', 'Library_code': 'PV033'},
 {'Taxon_code': '000000000000', 'Library_code': 'PV175'},
 {'Taxon_code': '000000000000', 'Library_code': 'PV196'},
 {'Taxon_code': '000000000000', 'Library_code': 'PV212'},
 {'Taxon_code': '000000000001', 'Library_code': 'PV002'},
 {'Taxon_code': '000000000001', 'Library_code': 'PV031'},
 {'Taxon_code': '000000000001', 'Library_code': 'PV079'},
 {'Taxon_code': '000000000001', 'Library_code': 'PV097'},
 {'Taxon_code': '000000000001', 'Library_code': 'PV132'},
 {'Taxon_code': '000000000001', 'Library_code': 'PV142'},
 {'Taxon_code': '000000000001', 'Library_code': 'PV154'},
 {'Taxon_code': '000000000001', 'Library_code': 'PV161'},
 {'Taxon_code': '000000000001', 'Library_code': 'PV188'},
 {'Taxon_code': '000000000001', 'Library_code': 'PV201'},
 {'Taxon_code': '000000000001', 'Library_code': 'PV214'},
 {'Taxon_code': '000000000001', 'Library_code': 'PV226'},
 {'Taxon_code'

In [21]:
host_library = dict(
    host=hosts.to_dict(orient='records'),
    host_library=hosts_library
)
with open("scratch/semantic-model/host_library.json", "w") as f:
    json.dump(host_library, f, indent=4, sort_keys=True)

## Step 3 - Virus

The file provided by Fernando requires some actions to turn it from its matrix form into a list of Library - Otu, which will be much easier to analyze later.

In [22]:
def extract_taxonomy_from_gene_records(host_name):
    url = "https://api.ncbi.nlm.nih.gov/datasets/v2"
    url = f'{url}/gene/accession/{host_name}/dataset_report'.replace(' ', '%20')
    r = requests.get(url)
    
    try: 
        r.raise_for_status()
    except requests.HTTPError:
        return pd.NA
    try:
        u = str(r.json()['reports'][0]['gene']['tax_id'])
    except KeyError:
        return pd.NA
    except IndexError:
        return pd.NA
    return u


In [23]:
virus = pd.read_csv("data/mcleish24-TableS4.csv", sep=';')
virus['OTU_taxid'] = virus['NCBI_accession'].apply(extract_taxonomy_from_gene_records)
virus

,OTU name,Family,NCBI_title,OTU_reference,Genome,%_ID_mean,%_ID_min,%_ID_max,BLAST_matches,NCBI_accession,OTU_taxid
0,Alfamovirus 1,Bromoviridae,Alfalfa mosaic virus RNA 1,AMV1,(+)ssRNA,96.6,94.4,99.2,4,NC_001495,12321
1,Alfamovirus 1,Bromoviridae,Alfalfa mosaic virus RNA 2,AMV2,(+)ssRNA,94.1,94.1,94.1,2,NC_002024,12321
2,Alphaendornavirus 1,Endornaviridae,Cucumis melo endornavirus,CmEV,dsRNA,96.5,87.9,100.0,4174,NC_029064,1776177
3,Alphaendornavirus 2,Endornaviridae,Bell pepper endornavirus,BPEV,dsRNA,100.0,100.0,100.0,2,NC_015781,<NA>
4,Alphaendornavirus 3,Endornaviridae,Hordeum vulgare endornavirus,HvEV,dsRNA,96.6,87.2,100.0,4002,NC_028949,1774276
...,...,...,...,...,...,...,...,...,...,...,...
172,Comovirus,Secoviridae,Arabidopsis latent virus-1 RNA1,ALaV1,(+)ssRNA,99.5,97.3,100.0,52,MH899120,<NA>
173,Comovirus,Secoviridae,Arabidopsis latent virus-1 RNA2,ALaV2,(+)ssRNA,97.6,94.0,100.0,30,MH899121,<NA>
174,Varicosavirus 1,Rhabdoviridae,Lettuce big-vein associated virus RNA 1,LBVaV1,(-)ssRNA,95.3,93.3,97.3,12,NC_011558,1985698
175,Varicosavirus 1,Rhabdoviridae,Lettuce big-vein associated virus RNA 2,LBVaV2,(-)ssRNA,94.9,90.6,98.0,32,NC_011568,1985698


In [24]:
virus_subset = virus[['OTU_taxid', 'NCBI_title', 'OTU_reference', 'NCBI_accession']].rename(columns={'OTU_taxid': 'taxid', 'NCBI_title': 'Scientific_name', 'OTU_reference': 'label'})
virus_subset['Taxon_code'] = [taxonids() for _ in range(len(virus_subset))]

virus_subset

,taxid,Scientific_name,label,NCBI_accession,Taxon_code
0,12321,Alfalfa mosaic virus RNA 1,AMV1,NC_001495,000000000000
1,12321,Alfalfa mosaic virus RNA 2,AMV2,NC_002024,000000000001
2,1776177,Cucumis melo endornavirus,CmEV,NC_029064,000000000002
3,<NA>,Bell pepper endornavirus,BPEV,NC_015781,000000000003
4,1774276,Hordeum vulgare endornavirus,HvEV,NC_028949,000000000004
...,...,...,...,...,...
172,<NA>,Arabidopsis latent virus-1 RNA1,ALaV1,MH899120,000000000172
173,<NA>,Arabidopsis latent virus-1 RNA2,ALaV2,MH899121,000000000173
174,1985698,Lettuce big-vein associated virus RNA 1,LBVaV1,NC_011558,000000000174
175,1985698,Lettuce big-vein associated virus RNA 2,LBVaV2,NC_011568,000000000175


In [25]:
hits = pd.read_csv("data/val_158_otu_host_df.csv", sep=';')
hits = hits.melt(
    id_vars=['Library', 'Host', 'Host_family', 'Habitat', 'Site', 'Collection'],
    value_vars=hits.columns[6:], 
).rename(columns={'variable': "OTU", 'value': "hit"}).query('hit == 1').copy()
hits['Collection'] = hits['Collection'].apply(lambda x: x.split("_")[0])
hits[['Library', 'Host', 'Host_family', 'Habitat', 'Site', 'Collection', 'OTU']]


,Library,Host,Host_family,Habitat,Site,Collection,OTU
74,PV091,Galium verum,Rubiaceae,Oak,Q4,Q4P,AEV1
332,PV059,Tragopogon sp,Asteraceae,Wasteland,E4,E4P,AEYV
425,PV159,Carduus bourgeanus,Asteraceae,Edge,L1,L1V,AEYV
448,PV182,Lactuca serriola,Asteraceae,Edge,L3,L3V,AEYV
537,PV498,Rubia peregrina,Rubiaceae,Crop,H3,H3P,AEYV
...,...,...,...,...,...,...,...
44458,PV172,Picris echioides,Asteraceae,Edge,L2,L2V,YSV
44487,PV201,Convolvulus arvensis,Convolvulaceae,Edge,L1,L1F,YSV
44625,PV047,Zea mays,Poaceae,Crop,Z2,Z2V,ZYMV
44638,PV061,Cucumis melo,Cucurbitaceae,Crop,M1,M5V,ZYMV


In [26]:
rename_map = pd.read_csv("scratch/mapping-otus-curated.csv", sep="\t", index_col=0)[['otu', 'hit']].set_index('otu').to_dict()['hit']
hits['OTU_rn'] = hits['OTU'].apply(lambda x: rename_map[x])
hits[hits['OTU_rn'].isna()][['OTU']].value_counts()

OTU 
BrYV    33
WYDV     4
FMV      1
Name: count, dtype: int64

In [27]:
hits = pd.merge(hits.dropna(subset=['OTU_rn'])[['Library', 'OTU_rn']], virus_subset[['label', 'Taxon_code']], how='left', right_on='label', left_on='OTU_rn')[['Library', 'Taxon_code']]
# hits[hits['OTU name'].isna()][['OTU']].value_counts()

In [28]:
hits

,Library,Taxon_code
0,PV091,000000000055
1,PV059,000000000082
2,PV159,000000000082
3,PV182,000000000082
4,PV498,000000000082
...,...,...
1573,PV172,000000000016
1574,PV201,000000000016
1575,PV047,000000000127
1576,PV061,000000000127


In [29]:
hits['Hit_code'] = [hitcodes() for _ in range(len(hits))]
hits[['Hit_code', 'Library', 'Taxon_code']]

,Hit_code,Library,Taxon_code
0,000000000000,PV091,000000000055
1,000000000001,PV059,000000000082
2,000000000002,PV159,000000000082
3,000000000003,PV182,000000000082
4,000000000004,PV498,000000000082
...,...,...,...
1573,000000001573,PV172,000000000016
1574,000000001574,PV201,000000000016
1575,000000001575,PV047,000000000127
1576,000000001576,PV061,000000000127


In [30]:
clean_dict = lambda d: {k: d[k] for k in d if not pd.isna(k)}

In [31]:
hits.query('Hit_code == "000000000111"')

,Library,Taxon_code,Hit_code
111,PV176,000000000044,000000000111


In [32]:
virus_library = dict(
    virus=list(map(clean_dict, virus_subset.to_dict(orient='records'))),
    hits=list(map(clean_dict, hits.to_dict(orient='records')))
)
with open("scratch/semantic-model/virus_library.json", "w") as f:
    json.dump(virus_library, f, indent=4, sort_keys=True)

## Step 3 - Bacteria

In [33]:
index_df = pd.read_csv("raw/motus-g1/source.csv", sep=",", header=None, index_col=0, names=['library', 'read1', 'read2'])
index_df

,library,read1,read2
1,PV001,PV1_18807_ATCACG_read1.fastq.gz,PV1_18807_ATCACG_read2.fastq.gz
2,PV002,PV2_18808_TGACCA_read1.fastq.gz,PV2_18808_TGACCA_read2.fastq.gz
3,PV003,PV3_16716_GGCTAC_read1.fastq.gz,PV3_16716_GGCTAC_read2.fastq.gz
4,PV003bgi,FCH5VT5ALXX_L5_HKRDPLAgshTAACRAAPEI-203_1.fq.zip,FCH5VT5ALXX_L5_HKRDPLAgshTAACRAAPEI-203_2.fq.zip
5,PV004bgi,FCH5VT5ALXX_L5_HKRDPLAgshTAADRAAPEI-204_1.fq.zip,FCH5VT5ALXX_L5_HKRDPLAgshTAADRAAPEI-204_2.fq.zip
...,...,...,...
320,PV586,PV586_03661AAB_AGTTGC_read1.fastq.gz,PV586_03661AAB_AGTTGC_read2.fastq.gz
321,PV587,PV587_03662AAB_ACCTTC_read1.fastq.gz,PV587_03662AAB_ACCTTC_read2.fastq.gz
322,PV588,PV588_03663AAB_TCGCAT_read1.fastq.gz,PV588_03663AAB_TCGCAT_read2.fastq.gz
323,PV589,PV589_03664AAB_AGGACT_read1.fastq.gz,PV589_03664AAB_AGGACT_read2.fastq.gz


In [34]:
def load_motus_classification(x, library):
    try:
        u = pd.read_csv(x, sep="\t")
    except pd.errors.EmptyDataError:
        u = pd.DataFrame(data=[], columns=['mOTU', 'taxonomy', 'taxid', 'abundance'])
    u.columns = ['mOTU', 'taxonomy', 'taxid', 'abundance']
    u = u.dropna(subset=['taxid'])
    u['library'] = library
    return u

In [35]:
bacteria = dict()
hits = []
for _, row in index_df.iterrows():
    
    u = load_motus_classification("raw/motus-g1/{:s}.classification.csv".format(row['library']), row['library'])
    
    if len(u) > 0:
        for _, item in u.iterrows():
            if item.taxid not in bacteria.keys():
                
                bacteria[item.taxid] = taxonids()
            taxon_code = bacteria[item.taxid]
            hits.append(
                {
                    'Hit_code': hitcodes(),
                    'Library': item.library,
                    'Taxon_code': taxon_code
                    
                }
            )


In [36]:
pd.DataFrame.from_records(hits)

,Hit_code,Library,Taxon_code
0,000000001578,PV001,000000000177
1,000000001579,PV002,000000000178
2,000000001580,PV002,000000000177
3,000000001581,PV002,000000000179
4,000000001582,PV003,000000000180
...,...,...,...
1499,000000003077,PV587,000000000317
1500,000000003078,PV588,000000000178
1501,000000003079,PV588,000000000699
1502,000000003080,PV589,000000000389


In [37]:
bacteria = pd.DataFrame(list(bacteria.items()), columns=['taxonid', 'Taxon_code'])
bacteria['taxonid'] = bacteria['taxonid'].astype(int)

In [38]:
import taxoniq

def fault_tolerant_name(x):
    try:
        return taxoniq.Taxon(x).scientific_name
    except KeyError:
        return ""

In [39]:
bacteria['scientific_name'] = bacteria['taxonid'].apply(lambda x: fault_tolerant_name(x))

In [40]:
bacteria

,taxonid,Taxon_code,scientific_name
0,2097,000000000177,Mycoplasmoides genitalium
1,59620,000000000178,uncultured Clostridium sp.
2,69896,000000000179,Candidatus Phytoplasma solani
3,641148,000000000180,Neisseria sp. oral taxon 014
4,1236179,000000000181,Janthinobacterium sp. B9-8
...,...,...,...
518,1835723,000000000695,Arthrobacter sp. OY3WO11
519,1955620,000000000696,Hymenobacter sp. CRA2
520,679249,000000000697,Holzapfeliella floricola
521,866801,000000000698,Apilactobacillus ozensis


In [41]:
from skbio.tree import TreeNode
tree = TreeNode.read("data/bac120.tree")
tree_tip_names = list(tree.subset())
tree_tip_names
gtdb_metadata = pd.read_csv("data/bac120_metadata.tsv", sep="\t")
gtdb_metadata[gtdb_metadata['gtdb_genome_representative'].apply(lambda x: x.replace("_", " ")).isin(tree_tip_names)]
gtdb_accessions = gtdb_metadata[['ncbi_taxid', 'gtdb_genome_representative']].drop_duplicates('ncbi_taxid', keep='first').copy()
gtdb_accessions['gtdb_genome_representative'] = gtdb_accessions['gtdb_genome_representative'].apply(lambda x: x.replace("_", " "))
# gtdb_accessions.to_csv('scratch/semantic-model/gtdb.csv', sep=',')
gtdb_accessions

,ncbi_taxid,gtdb_genome_representative
0,1331258,RS GCF 000657795.2
1,1282,RS GCF 006742205.1
2,2135698,RS GCF 000172415.1
3,90371,RS GCF 000006945.2
4,1236944,RS GCF 029023865.1
...,...,...
584333,1752224,GB GCA 001464955.1
584341,2665509,RS GCF 000006945.2
584345,2530375,RS GCF 004348725.1
584349,2169409,RS GCF 003097655.1


In [42]:
bacteria = pd.merge(bacteria, gtdb_accessions, left_on='taxonid', right_on='ncbi_taxid', how='left')

In [44]:
sanchis21 = pd.read_csv("data/sanchis21.tableS1.csv", sep="\t").drop_duplicates('TaxId', keep='first')
sanchis21['is_pab'] = True
sanchis21['pab_type'] = sanchis21.apply(lambda x: 'pab_pathogen' if x['PAB-phyto'] == 'P' else "pab_unknown", axis=1)
sanchis21['pab_type'] = sanchis21.apply(lambda x: 'pab_symbiont' if x['PAB-symb'] == 'S' else x['pab_type'], axis=1)
sanchis21

,TaxId,Biosample,Representative species,PAB-phyto,PAB-symb,is_pab,pab_type
0,178901,SAMN02849420,Acetobacter malorum,NaN,NaN,True,pab_unknown
1,441768,SAMN02603756,Acholeplasma laidlawii PG-8A,NaN,NaN,True,pab_unknown
2,32002,SAMN06198582,Achromobacter denitrificans,NaN,NaN,True,pab_unknown
3,217204,SAMN06174288,Achromobacter insolitus,NaN,NaN,True,pab_unknown
4,72556,SAMN03941583,Achromobacter piechaudii,NaN,NaN,True,pab_unknown
...,...,...,...,...,...,...,...
955,698414,SAMN06765826,Xylella fastidiosa subsp. pauca,P,NaN,True,pab_pathogen
956,1444770,SAMN02570451,Xylella taiwanensis,P,NaN,True,pab_pathogen
957,1735686,SAMN04151686,Xylophilus sp. Leaf220,NaN,NaN,True,pab_unknown
958,1288385,SAMEA1486427,Yersinia pekkanenii,NaN,NaN,True,pab_unknown


In [45]:
bacteria = pd.merge(bacteria, sanchis21[['TaxId', 'is_pab', 'pab_type']], left_on=['ncbi_taxid'], right_on=['TaxId'], how='left')

In [46]:
bacteria

,taxonid,Taxon_code,scientific_name,ncbi_taxid,gtdb_genome_representative,TaxId,is_pab,pab_type
0,2097,000000000177,Mycoplasmoides genitalium,NaN,NaN,NaN,NaN,NaN
1,59620,000000000178,uncultured Clostridium sp.,59620.0,GB GCA 900548855.1,NaN,NaN,NaN
2,69896,000000000179,Candidatus Phytoplasma solani,69896.0,RS GCF 000970375.1,NaN,NaN,NaN
3,641148,000000000180,Neisseria sp. oral taxon 014,641148.0,RS GCF 005886145.1,NaN,NaN,NaN
4,1236179,000000000181,Janthinobacterium sp. B9-8,1236179.0,RS GCF 000969645.2,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
518,1835723,000000000695,Arthrobacter sp. OY3WO11,1835723.0,RS GCF 001641125.1,1835723.0,True,pab_unknown
519,1955620,000000000696,Hymenobacter sp. CRA2,1955620.0,RS GCF 002007105.1,NaN,NaN,NaN
520,679249,000000000697,Holzapfeliella floricola,NaN,NaN,NaN,NaN,NaN
521,866801,000000000698,Apilactobacillus ozensis,866801.0,RS GCF 001435995.1,NaN,NaN,NaN


In [47]:
def clean_dict(x):
    new_dict = dict()
    for key, item in x.items():
        if not pd.isna(item):
            new_dict[key] = item
    return new_dict

In [48]:
bacteria_records = dict(
    taxon=list(map(clean_dict, bacteria.to_dict(orient='records'))),
    hits=hits
)




with open("scratch/semantic-model/bacteria_library.json", "w") as f:
    json.dump(bacteria_records, f, indent=4, sort_keys=True)

In [49]:
clean_dict(bacteria_records['taxon'][6])

{'taxonid': 29448,
 'Taxon_code': '000000000183',
 'scientific_name': 'Bradyrhizobium elkanii',
 'ncbi_taxid': 29448.0,
 'gtdb_genome_representative': 'RS GCF 023278185.1',
 'TaxId': 29448.0,
 'is_pab': True,
 'pab_type': 'pab_symbiont'}

In [50]:
bacteria_records['taxon']

[{'taxonid': 2097,
  'Taxon_code': '000000000177',
  'scientific_name': 'Mycoplasmoides genitalium'},
 {'taxonid': 59620,
  'Taxon_code': '000000000178',
  'scientific_name': 'uncultured Clostridium sp.',
  'ncbi_taxid': 59620.0,
  'gtdb_genome_representative': 'GB GCA 900548855.1'},
 {'taxonid': 69896,
  'Taxon_code': '000000000179',
  'scientific_name': 'Candidatus Phytoplasma solani',
  'ncbi_taxid': 69896.0,
  'gtdb_genome_representative': 'RS GCF 000970375.1'},
 {'taxonid': 641148,
  'Taxon_code': '000000000180',
  'scientific_name': 'Neisseria sp. oral taxon 014',
  'ncbi_taxid': 641148.0,
  'gtdb_genome_representative': 'RS GCF 005886145.1'},
 {'taxonid': 1236179,
  'Taxon_code': '000000000181',
  'scientific_name': 'Janthinobacterium sp. B9-8',
  'ncbi_taxid': 1236179.0,
  'gtdb_genome_representative': 'RS GCF 000969645.2'},
 {'taxonid': 1293,
  'Taxon_code': '000000000182',
  'scientific_name': 'Staphylococcus gallinarum',
  'ncbi_taxid': 1293.0,
  'gtdb_genome_representative'